# A5 H2 — GPU 사실 수집 (제출용 아님)

기존 H4 `company_size` 프롬프트·스키마로 dev 200건과 무라벨을 관측합니다.
H2 v11 부재 규칙의 **발화 건수·비율·dev 대비 배율**만 측정합니다.
새 Macro F1, 전체 v11 양성률, 서버 실행 통과를 측정하는 노트북이 아닙니다.

1. 런타임을 **A100 GPU**로 설정합니다. VRAM 30GiB 이상·여유 디스크 80GiB를 사전 검사합니다.
2. 원본 `train_unlabeled.jsonl`을 Drive의 `MyDrive/a5/`에 올립니다. 20,000건·SHA256을 확인합니다.
3. Colab 보안 비밀에 `HF_TOKEN`을 등록하고 노트북 접근을 허용합니다.
4. 위에서부터 모두 실행합니다. 기본은 dev 200건(처음만) + 다음 무라벨 1,000건입니다.
5. 끝나면 ZIP을 보관합니다. 같은 설정·Drive 폴더로 첫 셀부터 다시 실행하면 다음 묶음으로 이어집니다.

첫 회차 추론 약 25분, 무라벨 전체는 약 6.9시간이라는 **과거 A100 단순 환산**입니다.
설치·적재·입력 분포 차이는 별도입니다. 기본 1,000건씩 총 20회이며 `complete=true`가 되어야
20,000건 측정이 끝납니다. 중간 수치는 부분 표본입니다. 같은 폴더를 두 런타임에서 동시에 실행하지 마세요.

In [ ]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="a5-facts-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "c24f20865cdc00ae6e01f2efac170bcfacef0436"  # 검증한 A5 수집 코드, 변경하지 않음

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)

## 코드 고정

In [ ]:
REPO = WORK / "repo"
run_logged("git-clone", ["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
run_logged("git-fetch", ["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO)
run_logged("git-checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if SOURCE_COMMIT != REPO_REF:
    raise ValueError("REPO_REF를 실행 안내의 40자리 SHA로 고정하세요.")
SUBMISSION = REPO  # 기존 고정 환경 설치 셀에서 requirements.txt 경로로만 사용
write_json(RESULTS / "source.json", {"commit": SOURCE_COMMIT, "requested_ref": REPO_REF})
assert (REPO / "experiments/a5_collect_facts.py").is_file()

## Drive 입력 확인 — 업로드 후 이 셀부터 재실행 가능

무라벨 원본은 Git clone에 포함되지 않습니다. PC의 저장소 `open/train_unlabeled.jsonl`(약 791MB)을 Google Drive 웹 화면에서 **내 드라이브 → a5 폴더**에 업로드하세요. 업로드가 끝나면 아래 셀을 실행합니다. 이미 다른 Drive 폴더에 있다면 `UNLABELED_PATH`를 실제 경로로 바꾸세요. 파일 이름이나 확장자를 바꾸지 않습니다.

입력이 없어서 멈췄다면 코드 고정 셀을 반복할 필요 없이 **이 셀부터** 이어서 실행하세요. 아직 모델 설치·추론 전입니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
UNLABELED_PATH = Path("/content/drive/MyDrive/a5/train_unlabeled.jsonl")
FACTS_OUTPUT = Path("/content/drive/MyDrive/a5/h2-facts-" + SOURCE_COMMIT[:12])
SHARDS_PER_EPISODE = 1  # 다음 미완료 1,000건. 20이면 남은 전체를 시도하되 시간 예상 초과 시 중단.
if not UNLABELED_PATH.is_file():
    write_json(RESULTS / "input-check.json", {"status": "missing", "path": str(UNLABELED_PATH)})
    raise FileNotFoundError(f"입력 파일이 없습니다: {UNLABELED_PATH}\nPC의 open/train_unlabeled.jsonl을 Drive의 내 드라이브/a5 폴더에 업로드한 뒤 이 셀만 다시 실행하세요. 다른 위치에 있다면 UNLABELED_PATH를 수정하세요.")
with UNLABELED_PATH.open("rb") as f:
    input_hash = hashlib.file_digest(f, "sha256").hexdigest()
assert input_hash == "f46449fb84f980a5ddf868d66f5daff2bcf0991135d9b81da5656dd607275698", "원본 입력 SHA256 불일치"
with UNLABELED_PATH.open(encoding="utf-8") as f:
    assert sum(bool(line.strip()) for line in f) == 20000
write_json(RESULTS / "input-check.json", {"status": "verified", "path": str(UNLABELED_PATH), "sha256": input_hash, "count": 20000})
print("고정 코드:", SOURCE_COMMIT, "저장/재개:", FACTS_OUTPUT)

## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.

In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")

## 고정 Python·추론 패키지 설치

기존 Colab 검증과 같은 Python 3.12.13·vLLM 0.26.0·CUDA 13.0을 사용합니다.

In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 같은 저장소의 requirements.txt를 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")

## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.

In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})

## 기존 company_size 사실 수집

원응답·실제 문서 예산·환경·코드/입력 해시를 저장합니다. 1,000건 묶음 완료 시 Drive에 기록하며,
실패한 묶음만 다음 실행에서 다시 수집합니다. 코드·입력·환경이 다르면 기존 결과에 섞지 않습니다.
요약 `unlabeled.count`가 20,000 미만이면 전체 발화율 검증을 끝낸 것이 아닙니다.
7,200초 검사는 측정된 처리속도에 25% 여유를 붙인 회차 예상치이며 하드 타임아웃은 아닙니다.
이는 GPU 수집 운영 예산이며 서버 1,853건 시간 측정과 다릅니다. 서버 추가 호출은 0회입니다.

In [ ]:
inference_env = dict(os.environ)
inference_env.pop("HF_TOKEN", None)
inference_env.pop("HUGGING_FACE_HUB_TOKEN", None)
run_logged("a5-facts", [PYTHON, str(REPO / "experiments/a5_collect_facts.py"),
    "--input", str(UNLABELED_PATH), "--output-dir", str(FACTS_OUTPUT),
    "--model-dir", MODEL_DIR, "--shards-per-episode", str(SHARDS_PER_EPISODE)],
    env=inference_env, cwd=REPO)
summary = json.loads((FACTS_OUTPUT / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("20,000건 완료" if summary["complete"] else "부분 수집 완료. 같은 설정으로 재실행하면 다음 묶음부터 이어집니다.")

## 결과 ZIP 다운로드 — 실패했어도 이 셀 실행

Drive에는 완료 묶음과 원응답 이벤트가 이미 저장됩니다. 이 ZIP은 설치/실행 로그와 현재까지의
사실 전체를 묶습니다. ZIP을 전달하면 H2 발화율과 발화 사례를 검토할 수 있습니다.
중간 요약을 20,000건 완료나 채택으로 읽지 마세요. 제출 ZIP을 만들거나 대회에 제출하지 않습니다.

In [ ]:
from google.colab import files

archive_path = WORK / ("a5-facts-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, "logs/" + str(path.relative_to(RESULTS)))
    if "FACTS_OUTPUT" in globals() and FACTS_OUTPUT.exists():
        for path in sorted(FACTS_OUTPUT.rglob("*")):
            if path.is_file() and not path.name.endswith(".partial"):
                archive.write(path, "facts/" + str(path.relative_to(FACTS_OUTPUT)))
files.download(str(archive_path))